# Arquetipos de jugadores NBA por características físicas (Combine)

## Librerías

Cargamos todas las herramientas que vamos a necesitar:
- **pandas / numpy**: manipulación de datos
- **matplotlib / seaborn**: gráficas
- **StandardScaler**: normaliza las medidas físicas para que ninguna domine al resto (ej. el peso no aplaste a la altura)
- **KMeans**: el algoritmo que agrupa a los jugadores por similitud física
- **PCA**: reduce las 7 medidas a 2 dimensiones para poder visualizarlo en una gráfica

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

## Carga de datos

Cargamos dos archivos:
- **combine_final.csv**: medidas físicas de jugadores en el NBA Draft Combine (2000–2026). Altura, peso, envergadura, salto, agilidad, etc.
- **nbaplayersdraft_limpio.csv**: resultados reales del draft + carrera NBA de cada jugador (puntos, rebotes, asistencias, años jugados...)

El objetivo es usar el combine para crear grupos de jugadores similares, y luego cruzarlo con el historial para ver qué tipo de carrera tuvo cada arquetipo.

In [ ]:
combine = pd.read_csv('../datos/procesados/combine_final.csv')
draft   = pd.read_csv('../datos/procesados/nbaplayersdraft_limpio.csv')

print('Combine shape:', combine.shape)
print('Draft shape:', draft.shape)
print()
print(combine.dtypes)
print()
print(combine.isnull().sum())

## Limpieza y selección de columnas

Seleccionamos solo las 7 medidas físicas más fiables y relevantes:
altura, peso, envergadura, alcance en pie, salto máximo, agilidad y velocidad.

Eliminamos filas con valores a 0 (datos faltantes codificados como cero) y nos quedamos con una sola entrada por jugador (la más reciente si participó en varios combines).

In [ ]:
FEATURES = [
    'HEIGHT_W_SHOES',
    'WEIGHT',
    'WINGSPAN',
    'STANDING_REACH',
    'MAX_VERTICAL_LEAP',
    'LANE_AGILITY_TIME',
    'THREE_QUARTER_SPRINT',
]

# Filtramos filas donde todas las features tengan valor > 0
df = combine[['SEASON', 'PLAYER_NAME', 'POSITION'] + FEATURES].copy()
df = df[(df[FEATURES] > 0).all(axis=1)]
df = df.drop_duplicates(subset='PLAYER_NAME', keep='last')  # nos quedamos con la entrada más reciente

print(f'Jugadores válidos para clustering: {len(df)}')
df[FEATURES].describe().round(2)

## Método del codo — ¿cuántos grupos creamos?

KMeans necesita que le digamos de antemano cuántos grupos (clusters) queremos. Para elegir bien usamos el **método del codo**:

Probamos de 2 a 12 clusters y medimos la **inercia** (qué tan lejos están los jugadores de su centroide). Cuanto más clusters, menos inercia, pero llega un punto donde añadir más no mejora mucho. Ese "codo" en la gráfica es el número óptimo de clusters.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])

inercias = []
k_range = range(2, 13)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inercias.append(km.inertia_)

plt.figure(figsize=(9, 4))
plt.plot(k_range, inercias, 'o-', color='steelblue')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.title('Método del codo')
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Entrenamiento KMeans

Entrenamos el modelo con el número de clusters elegido en el codo (K=7).

KMeans agrupa a los jugadores de forma que los del mismo grupo se parezcan físicamente entre sí y sean distintos a los de otros grupos. Al final cada jugador tiene asignado un número de cluster (0 a 6).

In [ ]:
K = 7  # ajusta aquí si el codo sugiere otro valor

km = KMeans(n_clusters=K, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X)

print('Jugadores por cluster:')
print(df['cluster'].value_counts().sort_index())

## Visualización de los clusters (PCA)

Tenemos 7 medidas físicas por jugador, imposible de ver en una gráfica. Con **PCA** las comprimimos a 2 dimensiones manteniendo la mayor varianza posible.

El resultado es un mapa donde cada punto es un jugador y los colores son los clusters. Si los grupos están bien separados en el gráfico, el clustering está funcionando bien.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
df['pca1'] = coords[:, 0]
df['pca2'] = coords[:, 1]

plt.figure(figsize=(10, 7))
for c in range(K):
    mask = df['cluster'] == c
    plt.scatter(df.loc[mask, 'pca1'], df.loc[mask, 'pca2'], label=f'Cluster {c}', alpha=0.6, s=30)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
plt.title('Clusters de arquetipos físicos (PCA)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Varianza explicada por 2 componentes: {pca.explained_variance_ratio_.sum()*100:.1f}%')

## Análisis de centroides — ¿qué hay en cada cluster?

El **centroide** es el jugador "promedio" de cada cluster. Analizando sus medidas entendemos qué tipo de jugador representa cada grupo.

El heatmap normaliza los valores de 0 a 1 para comparar fácilmente: **rojo oscuro = valor alto**, **amarillo = valor bajo**. 

Recuerda que en LANE_AGILITY_TIME y THREE_QUARTER_SPRINT, **menos tiempo = más rápido**, así que amarillo aquí es bueno.

In [ ]:
centroides = df.groupby('cluster')[FEATURES].mean().round(2)
print(centroides.to_string())

# Heatmap de centroides normalizados
centroides_norm = (centroides - centroides.min()) / (centroides.max() - centroides.min())
plt.figure(figsize=(10, 5))
sns.heatmap(centroides_norm, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('Centroides de clusters (normalizado 0-1)')
plt.tight_layout()
plt.show()

## Nombres de los arquetipos

Mirando el heatmap anterior asignamos un nombre descriptivo a cada cluster. Estos nombres los usaremos en todo el análisis posterior para que sea más fácil de interpretar.

In [ ]:
NOMBRES_CLUSTERS = {
    0: 'Ala-Pívot tradicional',
    1: 'Base pequeño',
    2: 'Alero versátil',
    3: 'Escolta / Base técnico',
    4: 'Pívot élite moderno',
    5: 'Pívot clásico',
    6: 'Alero atlético explosivo',
}

df['arquetipo'] = df['cluster'].map(NOMBRES_CLUSTERS)

for c in range(K):
    jugadores = df[df['cluster'] == c]['PLAYER_NAME'].sample(min(5, (df['cluster'] == c).sum()), random_state=42).tolist()
    print(f"Cluster {c} — {NOMBRES_CLUSTERS[c]}: {jugadores}")

## Guardamos el modelo

Guardamos el modelo KMeans, el scaler y el PCA en la carpeta `modelos/` para poder reutilizarlos después sin tener que reentrenar. Son los tres elementos necesarios para predecir el arquetipo de cualquier jugador nuevo.

In [ ]:
import joblib, os

os.makedirs('../modelos', exist_ok=True)

joblib.dump(km,     '../modelos/kmeans_arquetipos.pkl')
joblib.dump(scaler, '../modelos/scaler_combine.pkl')
joblib.dump(pca,    '../modelos/pca_combine.pkl')

print("Modelos guardados:"
      "\n  modelos/kmeans_arquetipos.pkl"
      "\n  modelos/scaler_combine.pkl"
      "\n  modelos/pca_combine.pkl")

## Jugadores NBA de referencia por arquetipo

Asignamos a mano jugadores conocidos de los últimos años que representan bien cada arquetipo. Esto nos permite decir, por ejemplo: "este prospecto se parece físicamente a jugadores del estilo de Giannis o LeBron".

In [ ]:
# Jugadores de referencia por arquetipo — EDITAR A MANO
REFERENTES = {
    'Ala-Pívot tradicional':     ['Kevin Love', 'LaMarcus Aldridge', 'Blake Griffin', 'Draymond Green', 'Tobias Harris'],
    'Base pequeño':               ['Chris Paul', 'Isaiah Thomas', 'Kemba Walker', 'Trae Young', 'Kyle Lowry'],
    'Alero versátil':             ['Jaylen Brown', 'Khris Middleton', 'Jimmy Butler', 'Brandon Ingram', 'Mikal Bridges'],
    'Escolta / Base técnico':     ['Devin Booker', 'Bradley Beal', 'Klay Thompson', 'CJ McCollum', 'Shai Gilgeous-Alexander'],
    'Pívot élite moderno':        ['Anthony Davis', 'Bam Adebayo', 'Joel Embiid', 'Nikola Jokic', 'Karl-Anthony Towns'],
    'Pívot clásico':              ['Dwight Howard', 'Rudy Gobert', 'DeAndre Jordan', 'Brook Lopez', 'Boban Marjanovic'],
    'Alero atlético explosivo':   ['Giannis Antetokounmpo', 'LeBron James', 'Donovan Mitchell', 'Ja Morant', 'Zion Williamson'],
}

print("Jugadores de referencia por arquetipo:")
print("=" * 55)
for arquetipo, jugadores in REFERENTES.items():
    print(f"\n{arquetipo}:")
    for j in jugadores:
        print(f"  • {j}")

## Unión del combine con el historial NBA

Cruzamos los datos del combine con el historial real de draft para que cada jugador tenga tanto sus medidas físicas como sus estadísticas de carrera. El join se hace por nombre de jugador (en minúsculas para evitar errores de mayúsculas).

In [ ]:
# Normalizamos nombres para el join
df['nombre_norm'] = df['PLAYER_NAME'].str.lower().str.strip()
draft['nombre_norm'] = draft['player'].str.lower().str.strip()

df_merge = df.merge(draft, on='nombre_norm', how='inner')

print(f'Jugadores con combine + historial NBA: {len(df_merge)}')
print(df_merge[['PLAYER_NAME', 'cluster', 'arquetipo', 'points_per_game', 'years_active', 'overall_pick']].head(10))

## Estadísticas medias por arquetipo en la NBA

Calculamos los promedios de puntos, rebotes, asistencias y temporadas jugadas para cada arquetipo. Así podemos ver qué perfil físico tiende a tener mejor carrera en la NBA.

In [ ]:
# Solo jugadores con carrera real (al menos 1 temporada)
activos = df_merge[df_merge['years_active'] >= 1].copy()

stats_arquetipo = activos.groupby('arquetipo')[['points_per_game', 'average_total_rebounds', 
                                                  'average_assists', 'years_active']].mean().round(2)
stats_arquetipo = stats_arquetipo.sort_values('points_per_game', ascending=False)
print(stats_arquetipo)

## Función principal: buscar el arquetipo de un jugador

Dado el nombre de cualquier jugador que aparezca en el combine, esta función:
1. Identifica a qué cluster pertenece según sus medidas físicas
2. Muestra sus medidas
3. Lista los jugadores NBA de referencia de ese arquetipo
4. Busca en el historial los jugadores físicamente más parecidos y muestra su carrera real

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

def buscar_arquetipo(nombre_jugador, top_n=5):
    nombre_norm = nombre_jugador.lower().strip()
    match = df[df['nombre_norm'] == nombre_norm]

    if match.empty:
        match = df[df['nombre_norm'].str.contains(nombre_norm, na=False)]
        if match.empty:
            print(f"Jugador '{nombre_jugador}' no encontrado en el combine.")
            return

    jugador = match.iloc[0]
    cluster_id = jugador['cluster']
    arquetipo = jugador['arquetipo']

    print(f"\n{'='*60}")
    print(f"Jugador : {jugador['PLAYER_NAME']}")
    print(f"Temporada combine: {jugador['SEASON']}")
    print(f"Posición declarada: {jugador['POSITION']}")
    print(f"Arquetipo: {arquetipo}")
    print(f"{'='*60}")

    print("\nMedidas físicas:")
    for feat in FEATURES:
        print(f"  {feat}: {jugador[feat]:.2f}")

    # Referentes manuales del arquetipo
    refs = REFERENTES.get(arquetipo, [])
    if refs:
        print(f"\nJugadores NBA de referencia para este arquetipo:")
        for r in refs:
            print(f"  • {r}")

    # Comparables históricos del dataset (mismo cluster, carrera NBA)
    comparables = df_merge[
        (df_merge['cluster'] == cluster_id) &
        (df_merge['years_active'] >= 3) &
        (df_merge['nombre_norm'] != nombre_norm)
    ].copy()

    if comparables.empty:
        print("\nNo hay comparables históricos con carrera NBA en este cluster.")
        return

    jugador_vec = scaler.transform(jugador[FEATURES].values.reshape(1, -1))
    comp_X = scaler.transform(comparables[FEATURES])
    comparables['distancia'] = euclidean_distances(jugador_vec, comp_X)[0]
    comparables = comparables.sort_values('distancia').head(top_n)

    print(f"\nTop {top_n} comparables históricos más cercanos (por medidas físicas):")
    print("-" * 60)
    cols = ['PLAYER_NAME', 'overall_pick', 'years_active', 'points_per_game',
            'average_total_rebounds', 'average_assists', 'distancia']
    print(comparables[cols].rename(columns={
        'PLAYER_NAME': 'Jugador',
        'overall_pick': 'Pick',
        'years_active': 'Temporadas',
        'points_per_game': 'PPG',
        'average_total_rebounds': 'RPG',
        'average_assists': 'APG',
        'distancia': 'Distancia'
    }).to_string(index=False))


# Ejemplo
buscar_arquetipo('Aday Mara')

## Prueba con prospectos del Draft 2025–2026

Cambia el nombre por cualquier jugador que aparezca en el combine para ver su arquetipo y comparables.

In [ ]:
# Cambia el nombre por cualquier jugador del combine
buscar_arquetipo('Cooper Flagg')

## Otro prospecto

In [ ]:
buscar_arquetipo('Dylan Harper')